# Day 4 — HOL 1: Build the Bronze Layer — All 4 Sources, Audit Columns
### GlobalMart Data Engineering · 12:00 PM – 1:00 PM

---

## What We Are Building

```
SOURCE (ADLS raw-data/)          AUTOLOADER          BRONZE TABLE (gbmart.bronze)
──────────────────────────────────────────────────────────────────────────────
raw-data/customers/*.csv     →   cloudFiles (csv)  →  gbmart.bronze.customers
raw-data/addresses/*.csv     →   cloudFiles (csv)  →  gbmart.bronze.addresses
raw-data/payments/*.csv      →   cloudFiles (csv)  →  gbmart.bronze.payments
raw-data/products/*.json     →   cloudFiles (json) →  gbmart.bronze.products

SOURCE (Postgres/Supabase)        LAKEFLOW CONNECT         BRONZE TABLE
──────────────────────────────────────────────────────────────────────────────
orders, order_items           →   CDC (Day 2 HOL 1)   →   gbmart.bronze.orders
                                                            gbmart.bronze.order_items
```

**This HOL builds the 4 Autoloader-sourced tables hands-on.** The 2 CDC-sourced tables (`orders`, `order_items`) were already built by Day 2's Lakeflow Connect pipeline — this session ends by verifying they're there, not rebuilding them.

## By the End of This HOL You Will Have

- 4 new Bronze Delta tables in Unity Catalog: `gbmart.bronze.customers`, `gbmart.bronze.addresses`, `gbmart.bronze.payments`, `gbmart.bronze.products`
- Every table carrying the standard audit columns from ILT 2: `_ingested_at`, plus `_source_file`
- Hands-on experience with **two file formats** in one HOL — CSV (customers/addresses/payments) and nested JSON (products) — using the exact same Autoloader pattern for both
- Confirmation that all 6 Bronze tables (4 built here + 2 from Day 2's CDC pipeline) are live and queryable

---

## Before You Start — Cost & Safety Note

> **Do not start, trigger, or run any Databricks pipeline, job, or SQL warehouse from the Databricks UI or CLI as part of this HOL beyond what's in these notebook cells.** Everything here runs as ordinary notebook cells on your already-running cluster — there is no separate pipeline/job to "kick off." If a cell asks you to wait for a streaming query, that query stops itself (`trigger(availableNow=True)`) once it's processed the available files — it does not run indefinitely.

In [ ]:
# ─── SETUP ──────────────────────────────────────────────────────────────────────
# No storage key, no spark.conf.set — the Unity Catalog external location
# (registered in Day 2 HOL 1) handles authentication automatically.

from pyspark.sql.functions import col, current_timestamp

CATALOG = "gbmart"
SCHEMA  = "bronze"

# Real, already-registered external location for this cohort's shared build:
EXTERNAL_LOCATION = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Catalog '{CATALOG}' schema '{SCHEMA}' ready.")

# ─── Verify each source folder is visible before ingesting ─────────────────────
for folder in ["customers", "addresses", "payments", "products"]:
    files = dbutils.fs.ls(f"{EXTERNAL_LOCATION}/{folder}/")
    print(f"\n{folder}/  ({len(files)} file(s))")
    for f in files:
        print(f"  {f.name}  ({f.size / 1024:.1f} KB)")

---
## Phase 1 — `customers` (CSV, the Worked Example)

Read the code below carefully — Phases 2 and 3 reuse this exact pattern with only the source folder and table name changed.

> **If you already did Day 3's HOL 2** (which used `customers`/`payments` to teach Autoloader checkpoint mechanics), this cell targets the exact same table and checkpoint. That's intentional, not a bug: re-running is safe — the checkpoint means no duplicate rows land — and this HOL stays self-contained for anyone starting fresh here. Day 3 taught *how* Autoloader tracks state; this HOL delivers the *complete* 4-source Bronze layer as the real pipeline milestone.

In [ ]:
# ─── Ingest customers → gbmart.bronze.customers ────────────────────────────────
# This is the worked example — the next three sources reuse this exact shape.

SOURCE_FOLDER   = "customers"
TABLE           = "customers"
TARGET_TABLE    = f"{CATALOG}.{SCHEMA}.{TABLE}"
SOURCE_PATH     = f"{EXTERNAL_LOCATION}/{SOURCE_FOLDER}/"
CHECKPOINT_PATH = f"{EXTERNAL_LOCATION}/_checkpoints/{TABLE}/"
SCHEMA_PATH     = f"{EXTERNAL_LOCATION}/_schemas/{TABLE}/"

customers_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      SCHEMA_PATH)      # saves inferred schema for reuse
    .option("cloudFiles.inferColumnTypes",    "true")           # proper types, not all-string
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # new source columns added automatically
    .option("header",                         "true")
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))     # which file this row came from
    .withColumn("_ingested_at", current_timestamp())            # when Bronze wrote this row
)

(
    customers_df.writeStream
    .format("delta")
    .outputMode("append")                        # Bronze is append-only — never overwrite
    .option("checkpointLocation", CHECKPOINT_PATH)  # tracks which files are already processed
    .option("mergeSchema", "true")
    .trigger(availableNow=True)                  # process what's there, then stop
    .toTable(TARGET_TABLE)                       # Unity Catalog managed table — no path needed
)

df = spark.table(TARGET_TABLE)
print(f"gbmart.bronze.customers — total rows: {df.count()}")
df.show(3, truncate=False)

---
## Phase 2 — `addresses` (CSV, With a Multi-Line Gotcha)

Same pattern as Phase 1, plus one real-world wrinkle: some `AddressLine1` values contain embedded line breaks, which a naive CSV parser would split into extra (wrong) rows.

In [ ]:
# ─── Ingest addresses → gbmart.bronze.addresses ────────────────────────────────
# Same pattern as customers, with ONE addition: AddressLine1 can contain
# embedded newlines (e.g. multi-line street addresses), which breaks a plain
# CSV parser. multiLine="true" + escape='"' handles that correctly.

SOURCE_FOLDER   = "addresses"
TABLE           = "addresses"
TARGET_TABLE    = f"{CATALOG}.{SCHEMA}.{TABLE}"
SOURCE_PATH     = f"{EXTERNAL_LOCATION}/{SOURCE_FOLDER}/"
CHECKPOINT_PATH = f"{EXTERNAL_LOCATION}/_checkpoints/{TABLE}/"
SCHEMA_PATH     = f"{EXTERNAL_LOCATION}/_schemas/{TABLE}/"

addresses_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes",    "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",                         "true")
    .option("multiLine",                      "true")   # ← handles embedded newlines
    .option("escape",                         '"')
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(
    addresses_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

df = spark.table(TARGET_TABLE)
print(f"gbmart.bronze.addresses — total rows: {df.count()}")
df.show(3, truncate=False)

---
## Phase 3 — `payments` (CSV, Same Pattern as Customers)

Same shape as Phase 1 — no new concepts, just applying the pattern a third time so it's fully automatic by the time you hit the format change in Phase 4. (Also revisits Day 3 HOL 2's `payments` checkpoint — same safe-to-rerun note as Phase 1 applies here.)

In [ ]:
# ─── Ingest payments → gbmart.bronze.payments ──────────────────────────────────
# Same pattern as customers — plain CSV, no special options needed this time.

SOURCE_FOLDER   = "payments"
TABLE           = "payments"
TARGET_TABLE    = f"{CATALOG}.{SCHEMA}.{TABLE}"
SOURCE_PATH     = f"{EXTERNAL_LOCATION}/{SOURCE_FOLDER}/"
CHECKPOINT_PATH = f"{EXTERNAL_LOCATION}/_checkpoints/{TABLE}/"
SCHEMA_PATH     = f"{EXTERNAL_LOCATION}/_schemas/{TABLE}/"

payments_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes",    "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",                         "true")
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(
    payments_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

df = spark.table(TARGET_TABLE)
print(f"gbmart.bronze.payments — total rows: {df.count()}")
df.show(3, truncate=False)

---
## Phase 4 — `products` (JSON, Nested — A Different File Format)

Unlike the first three sources, `products` arrives as **JSON with nested structure** — each record has a `specs` struct, a `tags` array, and a `supplier_info` struct. This is a deliberate contrast: same Autoloader mental model, different `cloudFiles.format`, and Bronze doesn't need to flatten anything to land it correctly.

In [ ]:
# ─── Ingest products (JSON, nested) → gbmart.bronze.products ──────────────────
# Same Autoloader shape as customers/addresses/payments — only the format and
# a couple of JSON-specific options change. This is deliberate: one mental
# model, applied to two different file formats.
#
# products.json has nested structure per record:
#   { "product_id": "PRD-00001",
#     "specs": {"warranty_months": 24, "color_options": ["Blue","Black"]},
#     "tags": ["electronics", "gadget"],
#     "supplier_info": {"supplier_id": "SUP-02", "name": "Greenwood Supplies"} }
#
# Autoloader preserves the nested structs/arrays as-is — Silver will flatten
# them later. Bronze's job is just to land the data faithfully.

SOURCE_FOLDER   = "products"
TABLE           = "products"
TARGET_TABLE    = f"{CATALOG}.{SCHEMA}.{TABLE}"
SOURCE_PATH     = f"{EXTERNAL_LOCATION}/{SOURCE_FOLDER}/"
CHECKPOINT_PATH = f"{EXTERNAL_LOCATION}/_checkpoints/{TABLE}/"
SCHEMA_PATH     = f"{EXTERNAL_LOCATION}/_schemas/{TABLE}/"

products_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "json")   # ← different from CSV sources
    .option("cloudFiles.schemaLocation",      SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes",    "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("multiLine",                      "true")   # ← JSON array spread across lines
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(
    products_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

df = spark.table(TARGET_TABLE)
print(f"gbmart.bronze.products — total rows: {df.count()}")
df.show(3, truncate=False)

In [ ]:
# ─── Explore products' nested schema ───────────────────────────────────────────
# Bronze preserves nested structs and arrays exactly as they arrived — Silver
# will decide how (or whether) to flatten them. For now, just confirm they're
# readable with dotted / EXPLODE-style access.

products_bronze = spark.table("gbmart.bronze.products")
products_bronze.printSchema()

# Access nested fields directly — no flattening needed to just read them
products_bronze.select(
    "product_id",
    "product_name",
    "specs.warranty_months",
    "specs.color_options",
    "supplier_info.name",
).show(5, truncate=False)

---
## Phase 5 — Verify the CDC-Sourced Tables (Built in Day 2)

`orders` and `order_items` are **not built in this HOL** — they arrive via the Lakeflow Connect CDC pipeline you set up in Day 2 HOL 1. This phase only confirms they landed correctly in the same `gbmart.bronze` schema, so you can see the full 6-table Bronze layer side by side.

In [ ]:
# ─── Final verification: row counts across all 6 Bronze tables ────────────────
# Confirms the 4 tables built in this HOL AND the 2 CDC tables built in Day 2
# are all present and queryable in the gbmart catalog.

bronze_tables = [
    "customers",     # built today — Autoloader (csv)
    "addresses",     # built today — Autoloader (csv, multiLine)
    "payments",      # built today — Autoloader (csv)
    "products",      # built today — Autoloader (json, nested)
    "orders",        # built Day 2 — Lakeflow Connect CDC
    "order_items",   # built Day 2 — Lakeflow Connect CDC
]

print(f"{'TABLE':<30} {'ROWS':>10}  STATUS")
print("-" * 60)
for table in bronze_tables:
    full_name = f"gbmart.bronze.{table}"
    try:
        count = spark.table(full_name).count()
        print(f"  {full_name:<28} {count:>10,}  OK")
    except Exception as e:
        # If orders/order_items aren't there yet, it means Day 2's pipeline
        # hasn't been run in this workspace — not a bug in today's HOL.
        print(f"  {full_name:<28} {'—':>10}  NOT FOUND ({str(e)[:40]})")

---
## Key Takeaways

1. **Same Autoloader pattern, two formats** — `cloudFiles.format` is the only thing that changes between CSV and JSON sources; checkpoint, schema location, and audit-column logic are identical
2. **Unity Catalog managed tables need no path bookkeeping** — `spark.table("gbmart.bronze.products")` just works once the external location is registered (Day 2 HOL 1)
3. **Nested JSON lands as-is in Bronze** — `specs.warranty_months`-style dotted access works directly on struct columns; Silver will decide whether/how to flatten
4. **Minimal audit columns** — `_ingested_at` + `_source_file` is enough; no `_batch_id` or `_source_system` needed per ILT 2's reasoning
5. **CDC tables aren't rebuilt here** — `orders`/`order_items` already exist from Day 2's Lakeflow Connect pipeline; Bronze HOLs for different pathways don't duplicate each other's work

---

## Discussion Questions

1. *Why does `products` use `cloudFiles.format = "json"` while the other three use `"csv"`? What would happen if you used the wrong format option?*
2. *The `addresses` source needs `multiLine = "true"` but `customers` doesn't. What does that option do, and why would a CSV need it?*
3. *If `gbmart.bronze.orders` already exists from Day 2, why does this HOL still query it in the final verification step instead of skipping it entirely?*
4. *A new field `warranty_extended: true` is added to next week's `products.json` drop. What happens on the next Autoloader run? What configuration makes that safe?*
5. *Why is `trigger(availableNow=True)` the right choice here instead of a continuously running stream?*